# Face Emotion: entrenamiento en Colab (GPU)

Entrena modelos de emociones con transfer learning (EfficientNet preentrenada en ImageNet) usando
`train_transfer.py` y el dataset `images.rar` de este repositorio.

**Pasos:**
1. **Entorno de ejecución → Cambiar tipo de entorno → GPU (T4)** → Guardar
2. Revisa la configuración del paso 3 (por defecto entrena el modelo **v4: EfficientNetB2 a 128 px**)
3. **Entorno de ejecución → Ejecutar todas**
4. Al final se descargan los modelos `.h5`. Cópialos a la carpeta del proyecto: la app usa
   automáticamente todos los `modelFEC_v4*.h5` junto con `modelFEC_v3.h5` como **ensamble**.

Tiempo aproximado: **~1 hora por modelo v4** en una T4. No cierres la pestaña mientras entrena.

## 1. Verificar GPU e instalar Keras 2 (`tf_keras`)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "⚠️ SIN GPU: activa Entorno de ejecución → Cambiar tipo de entorno → GPU"
# tf_keras con la misma versión menor que el TensorFlow de Colab (para guardar en .h5 compatible con la app)
!TFV=$(python -c "import tensorflow as tf; print('.'.join(tf.__version__.split('.')[:2]))"); pip install -q "tf_keras==${TFV}.*"

## 2. Descargar el repositorio y descomprimir el dataset

In [ ]:
%cd /content
!rm -rf Face_Emotion
!git clone --depth 1 https://github.com/RicardoGarfias/Face_Emotion.git
%cd /content/Face_Emotion
!unrar x -Y -inul images.rar ./
!for s in train validation; do echo "$s: $(find images/$s -type f | wc -l) imágenes"; done

## 3. Configuración

| Opción | v4 (por defecto) | v3 (el modelo anterior) |
|---|---|---|
| `BASE` | `"b2"` | `"b0"` |
| `IMG_SIZE` | `128` | `96` |

`SEEDS`: una semilla por modelo. Con `[1]` se entrena un modelo (~1 h); con `[1, 2]`, dos (~2 h).
Cada modelo extra se suma al ensamble (+precisión, pero la app va algo más lenta).

In [ ]:
BASE = "b2"        # b0, b1, b2 o b3
IMG_SIZE = 128     # tamaño al que se amplía la cara de 48x48
SEEDS = [1]        # una semilla por modelo del ensamble
EPOCHS_HEAD, EPOCHS_FT = 3, 20

## 4. Entrenar
- Fase 1: base congelada (solo se entrena la cabeza).
- Fase 2: fine-tuning de toda la red; se detiene antes si deja de mejorar.

Se usa *mixed precision* (float16 en la GPU, ~2× más rápido). El `.h5` final se guarda en float32 y sin el optimizador.

In [ ]:
%cd /content/Face_Emotion
for seed in SEEDS:
    out = f"modelFEC_v4_{BASE}_{IMG_SIZE}_s{seed}.h5"
    print(f"\n########## Entrenando {out} ##########")
    !python -u train_transfer.py --base {BASE} --img-size {IMG_SIZE} --seed {seed} --mixed-precision --head-epochs {EPOCHS_HEAD} --ft-epochs {EPOCHS_FT} --model {out} 2>&1 | grep -vE "WARNING|UserWarning|saving_api|oneDNN|absl|cpu_feature"

## 5. Evaluar el ensamble
Compara el modelo original, cada modelo por separado y el ensamble de todos (v3 + los nuevos), con TTA.

In [ ]:
%cd /content/Face_Emotion
models = " ".join(["modelFEC_v3.h5"] + [f"modelFEC_v4_{BASE}_{IMG_SIZE}_s{s}.h5" for s in SEEDS])
!python train_transfer.py --eval --model {models} 2>&1 | grep -vE "WARNING|UserWarning|oneDNN|absl|cpu_feature"

## 6. Descargar los modelos
Cópialos a la carpeta del proyecto, junto a `modelFEC_v3.h5`.

In [ ]:
from google.colab import files
for s in SEEDS:
    files.download(f"/content/Face_Emotion/modelFEC_v4_{BASE}_{IMG_SIZE}_s{s}.h5")